In [1]:
from pathlib import Path
import pandas as pd

# ===============================
# CONFIGURACIÓN
# ===============================
CARPETA = Path(r"C:\Users\luisf\IQ Tech\DashboardRotacion\Stock Odoo Cuatitlan")
ARCHIVO = CARPETA / "rotacion_inventario_base_dashboard_odoo_autoazur.xlsx"
HOJA = "ventas_conjunto_detalle"

# ===============================
# CARGA
# ===============================
df = pd.read_excel(ARCHIVO, sheet_name=HOJA)
df.columns = [str(c).strip() for c in df.columns]

# ===============================
# VALIDACIONES
# ===============================
requeridas = ["fecha", "equipo_ventas", "venta_total"]
for col in requeridas:
    if col not in df.columns:
        raise KeyError(f"Columna obligatoria no encontrada: {col}")

# ===============================
# NORMALIZACIÓN
# ===============================
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["venta_total"] = pd.to_numeric(df["venta_total"], errors="coerce").fillna(0)

if "cantidad" in df.columns:
    df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(0)
else:
    df["cantidad"] = 0

# ===============================
# LIMPIEZA DE CANAL
# ===============================
df["equipo_ventas"] = (
    df["equipo_ventas"]
    .astype(str)
    .str.strip()
    .replace(
        ["", "None", "nan", "NaN", "SIN ASIGNAR", "Sin asignar", "sin asignar"],
        pd.NA
    )
    .fillna("SIN CANAL")
)

# ===============================
# FILTRO ÚLTIMOS 3 MESES (90 días)
# ===============================
fecha_fin = df["fecha"].max().normalize()
fecha_inicio = fecha_fin - pd.Timedelta(days=89)

df_3m = df[
    df["fecha"].notna() &
    (df["fecha"] >= fecha_inicio) &
    (df["fecha"] <= fecha_fin)
].copy()

# ===============================
# AGRUPACIÓN POR CANAL
# ===============================
resumen = (
    df_3m
    .groupby("equipo_ventas", as_index=False)
    .agg(
        ventas_totales=("venta_total", "sum"),
        unidades_totales=("cantidad", "sum"),
    )
)

# ===============================
# PORCENTAJES (1 = 100%)
# ===============================
total_ventas = resumen["ventas_totales"].sum()
total_unidades = resumen["unidades_totales"].sum()

resumen["pct_ventas"] = (
    resumen["ventas_totales"] / total_ventas
    if total_ventas > 0 else 0
)

resumen["pct_unidades"] = (
    resumen["unidades_totales"] / total_unidades
    if total_unidades > 0 else 0
)

# ===============================
# ORDEN FINAL
# ===============================
resumen = (
    resumen
    .sort_values("ventas_totales", ascending=False)
    .reset_index(drop=True)
)

# ===============================
# RESULTADO
# ===============================
print(resumen.to_string(index=False))

# ===============================
# EXPORTAR
# ===============================
salida = CARPETA / "ventas_por_canal_ultimos_3_meses_con_porcentajes.xlsx"
with pd.ExcelWriter(salida, engine="openpyxl") as writer:
    resumen.to_excel(writer, sheet_name="ventas_por_canal_3m", index=False)

print(f"\nArchivo generado: {salida}")

equipo_ventas  ventas_totales  unidades_totales  pct_ventas  pct_unidades
Mercado Libre     19673510.67             16680    0.360606      0.373413
       Amazon     14028337.69             13225    0.257133      0.296067
      Walmart      9701804.75              4196    0.177829      0.093935
    Liverpool      6244581.68              6932    0.114460      0.155186
    SIN CANAL      2689219.62              1927    0.049292      0.043140
       Coppel      1833260.43               696    0.033603      0.015581
       TikTok       386109.84              1013    0.007077      0.022678

Archivo generado: C:\Users\luisf\IQ Tech\DashboardRotacion\Stock Odoo Cuatitlan\ventas_por_canal_ultimos_3_meses_con_porcentajes.xlsx
